In [161]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [162]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [163]:
df = pd.read_csv("/content/mumbai.csv")

In [164]:
df.sample(1)

,price,Address,area,latitude,longitude,Bedrooms,Bathrooms,Balcony,Status,neworold,parking,Furnished_status,Lift,Landmarks,type_of_building,desc,Price_sqft
4333,42500000.0,"Lokhandwala Complex, Andheri West, Mumbai, Maharashtra, Andheri West, Mumbai - Western Suburbs, Maharashtra",1650.0,19.142096,72.826173,3.0,3.0,NaN,Ready to Move,Resale,NaN,NaN,NaN,NaN,Individual House,\n,25757.575758


In [165]:
# Removing Unwanted Columns
# latitude and longitude ( Removing this for Now Because we are Building Prediction Model First. )

df.drop(
    columns = ["latitude", "longitude", "parking", "Lift", "Landmarks", "desc"],
    inplace=True
)

In [166]:
df.sample(1)

,price,Address,area,Bedrooms,Bathrooms,Balcony,Status,neworold,Furnished_status,type_of_building,Price_sqft
5971,22500000.0,"Lokhandwala Complex, Mumbai - Western Suburbs, Maharashtra",1000.0,2.0,2.0,NaN,Ready to Move,Resale,Unfurnished,Flat,22500.0


In [167]:
df['Address'] = df['Address'].str.lower()
df['Address'] = df['Address'].str.split("-").str.get(1)
df['Address'] = df['Address'].str.split(",").str.get(0)

def add_split(address):
  address = address.str.lower()
  address = address.str.split("-").str.get(1)
  address = address.str.split(",").str.get(0)

  return address

In [168]:
df = df[~df['Address'].isnull()]

In [169]:
df[df['Address'].str.lower().str.contains("around")].shape

(88, 11)

In [170]:
import pandas as pd
import re

# Specific
WEST_SPECIFIC = [
    "andheri west", "bandra west", "khar west", "santacruz west",
    "vile parle west", "jogeshwari west", "goregaon west", "malad west"
]

EAST_SPECIFIC = [
    "andheri east", "goregaon east", "malad east", "bandra east",
    "vile parle east", "central navi mumbai"
]

# General
NORTH_LOCALITIES = [
    "borivali", "dahisar", "kandivali", "malad", "mira road",
    "bhayandar", "vasai", "virar", "uttan", "thakur village",
    "akurli nagar", "kurar village", "dindoshi", "kasturpark",
    "gorai", "kashimira", "kalyan", "kalyan road", "charkop"
]

WEST_LOCALITIES = [
    "juhu", "versova", "lokhandwala", "oshiwara", "pali hill"
]

EAST_LOCALITIES = [
    "chembur", "ghatkopar", "vikhroli", "powai", "chandivali",
    "kalina", "sakinaka", "kurla", "kanjurmarg", "bhandup",
    "mulund", "govandi", "deonar", "nahur", "trombay",
    "chunabhatti", "tilak nagar", "central line", "harbour line",
    "around mumbai", "khargar", "beyond thane", "majiwada"
]

SOUTH_LOCALITIES = [
    "colaba", "fort", "churchgate", "cuffe parade", "marine lines",
    "nariman point", "csmt", "cst", "kalbadevi", "girgaon",
    "malabar hill", "walkeshwar", "breach candy", "tardeo",
    "grant road", "mumbai central", "byculla", "mazgaon",
    "lower parel", "parel", "worli", "prabhadevi",
    "n m joshi marg", "central mumbai"
]

# Validated PIN Code Matching
NORTH_PINS = {
    "400066", "400091", "400092", "400095", "400097", "400101",
    "400103", "400104", "401101", "401105", "401107", "421003",
    "400068", "401201", "401202"
}

WEST_PINS = {
    "400049", "400050", "400051", "400052", "400053", "400054",
    "400056", "400061", "400064", "400055", "400058", "400062"
}

EAST_PINS = {
    "400037", "400057", "400063", "400069", "400070", "400071",
    "400072", "400076", "400077", "400080", "400086", "400093",
    "400098", "400601", "421202"
}

SOUTH_PINS = {
    "400001", "400002", "400003", "400005", "400006", "400007",
    "400008", "400009", "400010", "400011", "400012", "400013",
    "400018", "400020", "400021", "400026"
}


def categorize_mumbai(address):
    if pd.isna(address):
        return address

    # Standardize string formatting
    address = str(address).lower().strip()
    address = re.sub(r"\s+", " ", address)

    # Regex search
    pin_match = re.search(r"(?<!\d)\d{6}(?!\d)", address)
    pin = pin_match.group() if pin_match else None

    # Step 1
    if any(x in address for x in WEST_SPECIFIC):
        return "West Mumbai"
    if any(x in address for x in EAST_SPECIFIC):
        return "East Mumbai"

    # Step 2
    if any(x in address for x in NORTH_LOCALITIES):
        return "North Mumbai"
    if any(x in address for x in WEST_LOCALITIES):
        return "West Mumbai"
    if any(x in address for x in EAST_LOCALITIES):
        return "East Mumbai"
    if any(x in address for x in SOUTH_LOCALITIES):
        return "South Mumbai"

    # Step 3
    if pin in NORTH_PINS:
        return "North Mumbai"
    if pin in WEST_PINS:
        return "West Mumbai"
    if pin in EAST_PINS:
        return "East Mumbai"
    if pin in SOUTH_PINS:
        return "South Mumbai"

    # Step 4
    if address in ["north", "north mumbai"]:
        return "North Mumbai"
    if address in ["west", "west mumbai", "western suburbs"]:
        return "West Mumbai"
    if address in ["east", "east mumbai"]:
        return "East Mumbai"
    if address in ["south", "south mumbai"]:
        return "South Mumbai"

    # Leave address
    return address

df["Category"] = df["Address"].apply(categorize_mumbai)


In [176]:
df = df[df['Category'].isin(["North Mumbai" , "South Mumbai" , "West Mumbai" , "East Mumbai"])]

In [177]:
df['Category'].value_counts()

,count
Category,
North Mumbai,1902
West Mumbai,1755
East Mumbai,1496
South Mumbai,701


In [181]:
# 1st Step
df.columns

df.rename(columns={
    'Address':'address',
    'Bedrooms':'bedrooms',
    'Bathrooms':'bathrooms',
    'Balcony':'balcony',
    'Status':'status',
    'Furnished_status':'furnished_status',
    'Price_sqft':'price_sqft',
    'Category':'category'
},
inplace=True
)

In [182]:
df.sample(10)

,price,address,area,bedrooms,bathrooms,balcony,status,neworold,furnished_status,type_of_building,price_sqft,category
226,25300000.0,western suburbs,1450.0,3.0,3.0,NaN,Ready to Move,New Property,NaN,Flat,17448.275862,West Mumbai
979,16500000.0,north mumbai,1660.0,3.0,3.0,1.0,Ready to Move,Resale,Furnished,Flat,9939.759036,North Mumbai
3138,4000000.0,mira road and beyond,735.0,2.0,2.0,NaN,Ready to Move,Resale,Furnished,Flat,5442.176871,North Mumbai
2863,5500000.0,mira road and beyond,995.0,2.0,2.0,3.0,Ready to Move,Resale,Semi-Furnished,Flat,5527.638191,North Mumbai
639,13000000.0,mira road and beyond,1500.0,3.0,3.0,1.0,Ready to Move,Resale,NaN,Flat,8666.666667,North Mumbai
4410,31000000.0,central line,1040.0,3.0,2.0,1.0,Ready to Move,Resale,NaN,Flat,29807.692308,East Mumbai
3902,8500000.0,mira road and beyond,920.0,2.0,2.0,NaN,Ready to Move,Resale,NaN,Flat,9239.130435,North Mumbai
1574,13500000.0,mira road and beyond,1100.0,2.0,2.0,4.0,Ready to Move,Resale,NaN,Flat,12272.727273,North Mumbai
5414,27500000.0,western suburbs,750.0,2.0,2.0,1.0,Ready to Move,Resale,Furnished,Flat,36666.666667,West Mumbai
1591,26000000.0,western suburbs,657.0,2.0,2.0,NaN,Ready to Move,New Property,NaN,Flat,39573.820396,West Mumbai


In [183]:
# step 2
df.drop(columns=['address'],inplace=True)

In [184]:
df.sample(1)

,price,area,bedrooms,bathrooms,balcony,status,neworold,furnished_status,type_of_building,price_sqft,category
433,5700000.0,855.0,2.0,2.0,2.0,Ready to Move,Resale,NaN,Flat,6666.666667,North Mumbai


In [201]:
def add_split(address):
  address = address.lower()
  address = address.split("-")[1] if "-" in address else address
  address = address.split(",")[0]

  return address.strip()

In [203]:
add_split("CSion, Mumbai - Central Mumbai, Maharashtra")

'central mumbai'